In [ ]:
from pathlib import Path
import re
import numpy as np
import tifffile
import napari

In [ ]:
def find_repo_root(start: Path = Path.cwd()) -> Path:
    start = start.resolve()

    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / "learned").exists():
            return path

    raise RuntimeError("Could not locate the repository root.")

PROJECT_ROOT = find_repo_root()

BLASTOSPIM_ROOT = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "BlastoSPIM1_sample"
)

SERIES = "F22"

pattern = re.compile(
    rf"^{SERIES}_(\d+)_image_(\d+)\.npy$"
)

frames = []

for image_path in BLASTOSPIM_ROOT.glob(f"{SERIES}_*_image_*.npy"):
    match = pattern.match(image_path.name)

    if match is None:
        continue

    timepoint = int(match.group(1))
    image_index = int(match.group(2))

    mask_path = (
        BLASTOSPIM_ROOT
        / f"{SERIES}_{timepoint:03d}_masks_{image_index:04d}.npy"
    )

    if not mask_path.exists():
        raise FileNotFoundError(
            f"Missing mask for {image_path.name}: {mask_path.name}"
        )

    frames.append(
        {
            "timepoint": timepoint,
            "image_index": image_index,
            "image_path": image_path,
            "mask_path": mask_path,
        }
    )

frames.sort(key=lambda x: (x["timepoint"], x["image_index"]))

print("Frames found:", len(frames))

for frame in frames:
    print(
        f"F22_{frame['timepoint']:03d}",
        "->",
        frame["image_path"].name,
        "|",
        frame["mask_path"].name,
    )

In [ ]:
first = frames[0]

image = np.load(first["image_path"], allow_pickle=False)
mask = np.load(first["mask_path"], allow_pickle=False)

print("Timepoint:", first["timepoint"])

print("\nImage")
print(" shape:", image.shape)
print(" dtype:", image.dtype)
print(" min:", image.min())
print(" max:", image.max())

print("\nMask")
print(" shape:", mask.shape)
print(" dtype:", mask.dtype)
print(" min:", mask.min())
print(" max:", mask.max())

instance_ids = np.unique(mask)
instance_ids = instance_ids[instance_ids != 0]

print(" instances:", len(instance_ids))
print(" IDs:", instance_ids)

assert image.ndim == 3
assert mask.ndim == 3
assert image.shape == mask.shape

In [ ]:
raw_frames = []
gt_frames = []
frame_numbers = []

expected_shape = None

for frame in frames:
    raw = np.load(frame["image_path"], allow_pickle=False)
    gt = np.load(frame["mask_path"], allow_pickle=False)

    if raw.shape != gt.shape:
        raise ValueError(
            f"Shape mismatch at F22_{frame['timepoint']:03d}: "
            f"raw={raw.shape}, gt={gt.shape}"
        )

    if expected_shape is None:
        expected_shape = raw.shape

    if raw.shape != expected_shape:
        raise ValueError(
            f"Inconsistent volume shape at F22_{frame['timepoint']:03d}: "
            f"{raw.shape} != {expected_shape}"
        )

    raw_frames.append(raw)
    gt_frames.append(gt)
    frame_numbers.append(frame["timepoint"])

raw_movie = np.stack(raw_frames, axis=0)
gt_movie = np.stack(gt_frames, axis=0)

print("Frame numbers:", frame_numbers)
print("Raw movie:", raw_movie.shape, raw_movie.dtype)
print("GT movie :", gt_movie.shape, gt_movie.dtype)

assert raw_movie.shape == gt_movie.shape
assert raw_movie.ndim == 4

In [ ]:
for t, frame_number in enumerate(frame_numbers):
    ids = np.unique(gt_movie[t])
    ids = ids[ids != 0]

    print(
        f"F22_{frame_number:03d}: "
        f"{len(ids)} instances"
    )

In [ ]:
%gui qt

In [ ]:
import napari

SPACING_ZYX_UM = (2.0, 0.208, 0.208)

lo, hi = np.percentile(raw_movie, [1.0, 99.8])

viewer = napari.Viewer(ndisplay=3)

raw_layer = viewer.add_image(
    raw_movie,
    name="F22 raw",
    colormap="gray",
    scale=(1.0, *SPACING_ZYX_UM),
    rendering="mip",
    contrast_limits=(float(lo), float(hi)),
)

gt_layer = viewer.add_labels(
    gt_movie,
    name="F22 GT instances",
    scale=(1.0, *SPACING_ZYX_UM),
    opacity=0.45,
)

viewer.dims.axis_labels = ("T", "Z", "Y", "X")
viewer.dims.ndisplay = 3
viewer.reset_view()

In [ ]:
from importlib import import_module

from src.api import (
    preprocess_volume,
    create_binary_mask,
)

PreprocessingConfig = import_module(
    "src.01_preprocessing.config"
).PreprocessingConfig

MaskingConfig = import_module(
    "src.02_masking.config"
).MaskingConfig


preprocessing_config = PreprocessingConfig(
    low_percentile=1.0,
    high_percentile=99.5,
    denoise_sigma_um=0.8,
    background_sigma_um=4.0,
    voxel_size_zyx_um=SPACING_ZYX_UM,
)

masking_config = MaskingConfig()

print(preprocessing_config)
print(masking_config)

In [ ]:
processed_frames = []
stage1_traces = []

for t in range(raw_movie.shape[0]):
    processed, trace = preprocess_volume(
        raw_movie[t],
        config=preprocessing_config,
        return_diagnostics=True,
    )

    processed_frames.append(processed)
    stage1_traces.append(trace)

processed_movie = np.stack(processed_frames, axis=0)

print("Raw movie      :", raw_movie.shape, raw_movie.dtype)
print("Processed movie:", processed_movie.shape, processed_movie.dtype)

print(
    "Processed range:",
    float(processed_movie.min()),
    float(processed_movie.max()),
)

In [ ]:
binary_frames = []
stage2_traces = []

for t in range(processed_movie.shape[0]):
    binary, trace = create_binary_mask(
        processed_movie[t],
        config=masking_config,
        return_diagnostics=True,
    )

    binary_frames.append(binary)
    stage2_traces.append(trace)

binary_movie = np.stack(binary_frames, axis=0)

print("Binary movie:", binary_movie.shape, binary_movie.dtype)

for t, frame_number in enumerate(frame_numbers):
    trace = stage2_traces[t]

    print(
        f"F22_{frame_number:03d} | "
        f"foreground={binary_movie[t].sum():,} | "
        f"fraction={binary_movie[t].mean():.3%} | "
        f"Otsu={trace.metrics['base_otsu_threshold']:.5f}"
    )

In [ ]:
from scipy import ndimage as ndi

CONNECTIVITY_6 = ndi.generate_binary_structure(3, 1)

instance_frames = []
component_counts = []

for t in range(binary_movie.shape[0]):
    labels, count = ndi.label(
        binary_movie[t],
        structure=CONNECTIVITY_6,
    )

    labels = labels.astype(np.int32, copy=False)

    instance_frames.append(labels)
    component_counts.append(int(count))

instance_movie = np.stack(instance_frames, axis=0)

print("Initial instances:", instance_movie.shape, instance_movie.dtype)

for t, frame_number in enumerate(frame_numbers):
    gt_count = len(np.unique(gt_movie[t])) - 1

    print(
        f"F22_{frame_number:03d} | "
        f"initial CC={component_counts[t]:3d} | "
        f"GT={gt_count:3d}"
    )

In [ ]:
for t, frame_number in enumerate(frame_numbers):
    labels = instance_movie[t]

    counts = np.bincount(labels.ravel())
    sizes = counts[1:]

    if len(sizes) == 0:
        print(f"F22_{frame_number:03d}: no components")
        continue

    print(
        f"F22_{frame_number:03d} | "
        f"N={len(sizes):3d} | "
        f"min={sizes.min():5d} | "
        f"median={np.median(sizes):8.1f} | "
        f"max={sizes.max():7d} | "
        f"<10 vox={np.sum(sizes < 10):3d} | "
        f"<50 vox={np.sum(sizes < 50):3d}"
    )

In [ ]:
for t, frame_number in enumerate(frame_numbers):
    pred_fg = instance_movie[t] > 0
    gt_fg = gt_movie[t] > 0

    intersection = np.logical_and(pred_fg, gt_fg).sum()
    union = np.logical_or(pred_fg, gt_fg).sum()

    tp = intersection
    fp = np.logical_and(pred_fg, ~gt_fg).sum()
    fn = np.logical_and(~pred_fg, gt_fg).sum()

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    iou = intersection / max(union, 1)

    print(
        f"F22_{frame_number:03d} | "
        f"precision={precision:.3f} | "
        f"recall={recall:.3f} | "
        f"IoU={iou:.3f}"
    )

In [ ]:
viewer.add_image(
    processed_movie,
    name="Stage 1 processed",
    colormap="gray",
    scale=(1.0, *SPACING_ZYX_UM),
    rendering="mip",
    visible=False,
)

viewer.add_labels(
    binary_movie.astype(np.uint8),
    name="Stage 2 foreground",
    scale=(1.0, *SPACING_ZYX_UM),
    opacity=0.35,
    visible=False,
)

viewer.add_labels(
    instance_movie,
    name="Initial instances - CC only",
    scale=(1.0, *SPACING_ZYX_UM),
    opacity=0.55,
)

viewer.add_labels(
    gt_movie,
    name="GT instances",
    scale=(1.0, *SPACING_ZYX_UM),
    opacity=0.45,
    visible=False,
)

In [ ]:
import numpy as np
from scipy import ndimage as ndi

markers_movie = np.zeros_like(instance_movie, dtype=np.int32)

for t in range(instance_movie.shape[0]):
    labels = instance_movie[t]

    # Bounding box of every labeled instance
    objects = ndi.find_objects(labels)

    for instance_id, slc in enumerate(objects, start=1):
        if slc is None:
            continue

        # Expand bounding box by 1 voxel, while staying inside image
        expanded = tuple(
            slice(
                max(0, s.start - 1),
                min(labels.shape[d], s.stop + 1),
            )
            for d, s in enumerate(slc)
        )

        local_labels = labels[expanded]
        component = local_labels == instance_id

        if not component.any():
            continue

        edt = ndi.distance_transform_edt(
            component,
            sampling=SPACING_ZYX_UM,
        )

        local_position = np.unravel_index(
            np.argmax(edt),
            edt.shape,
        )

        # Convert local crop coordinates -> full-volume coordinates
        global_position = tuple(
            expanded[d].start + local_position[d]
            for d in range(3)
        )

        markers_movie[t][global_position] = instance_id

print("Markers:", markers_movie.shape)
print(
    "Markers/frame:",
    [
        int(np.count_nonzero(markers_movie[t]))
        for t in range(markers_movie.shape[0])
    ],
)

In [ ]:
import json

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

np.save(
    OUTPUT_DIR / "frame_numbers.npy",
    np.asarray(frame_numbers, dtype=np.int32),
)

np.save(
    OUTPUT_DIR / "raw_movie.npy",
    raw_movie,
)

np.save(
    OUTPUT_DIR / "processed_movie.npy",
    processed_movie.astype(np.float32),
)

np.save(
    OUTPUT_DIR / "binary_movie.npy",
    binary_movie.astype(bool),
)

np.save(
    OUTPUT_DIR / "instance_movie.npy",
    instance_movie.astype(np.int32),
)

np.save(
    OUTPUT_DIR / "markers_movie.npy",
    markers_movie.astype(np.int32),
)

np.save(
    OUTPUT_DIR / "gt_movie.npy",
    gt_movie.astype(np.int32),
)

metadata = {
    "dataset": "BlastoSPIM1",
    "series": SERIES,
    "frame_numbers": [int(x) for x in frame_numbers],
    "spacing_zyx_um": [
        float(x) for x in SPACING_ZYX_UM
    ],
    "pipeline": {
        "stage_1": "canonical preprocessing",
        "stage_2": "canonical Otsu masking",
        "stage_3": (
            "6-connected-component initialization only; "
            "probabilistic peak reasoning, geometric completion "
            "and watershed disabled"
        ),
    },
    "trackastra": {
        "images": "processed_movie.npy",
        "masks": "instance_movie.npy",
    },
}

with open(
    OUTPUT_DIR / "metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )

print("Saved to:")
print(OUTPUT_DIR)

for path in sorted(OUTPUT_DIR.iterdir()):
    print(
        f"{path.name:25s}",
        f"{path.stat().st_size / 1024**2:8.2f} MB",
    )